# 04: Train Reranker on Colab GPU

This notebook is the GPU training stage for the Cross-Encoder reranker. S3/DVC is the source of truth; Google Drive can be used only as a local Colab cache.

## Runtime

Use `Runtime -> Change runtime type -> T4/A100` before running. The notebook pulls prepared artifacts from S3, trains the reranker, stores the checkpoint under `artifacts/experiments/<RUN_ID>/models/reranker`, then pushes artifacts back to S3 through DVC.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime -> Change runtime type -> T4/A100."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
REPO_OWNER = "terrylimax"
REPO_NAME = "medical-rag-reranker"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
BRANCH = os.getenv("PROJECT_BRANCH", "main")
PROJECT_DIR = Path("/content") / REPO_NAME

RUN_ID = os.getenv("EXPERIMENT_RUN_ID", "medquad_full_remote")
OUTPUT_DIR = Path("artifacts") / "experiments" / RUN_ID
RERANKER_DIR = OUTPUT_DIR / "models" / "reranker"
MLFLOW_TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "file:/content/drive/MyDrive/medical-rag-reranker-colab/mlruns",
)
os.environ.setdefault("MEDICAL_RAG_PROGRESS", "0")

print("RUN_ID:", RUN_ID)
print("Output dir:", OUTPUT_DIR)
print("MLflow tracking URI:", MLFLOW_TRACKING_URI)
print("Project tqdm progress:", os.getenv("MEDICAL_RAG_PROGRESS"))

In [ ]:
def sh(*args, cwd=None):
    args = [str(arg) for arg in args]
    print("+", " ".join(args), flush=True)
    subprocess.run(args, cwd=None if cwd is None else str(cwd), check=True)


def run_streaming(*args, cwd=None):
    args = [str(arg) for arg in args]
    print("+", " ".join(args), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        args,
        cwd=None if cwd is None else str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0,
    )
    assert proc.stdout is not None
    while True:
        chunk = proc.stdout.read(1)
        if chunk == "" and proc.poll() is not None:
            break
        if chunk:
            print(chunk, end="", flush=True)
    returncode = proc.wait()
    if returncode != 0:
        raise RuntimeError(f"Command failed with exit code {returncode}")


if MLFLOW_TRACKING_URI.startswith("file:/content/drive/"):
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", type(exc).__name__, str(exc))


if not PROJECT_DIR.exists():
    sh("git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=PROJECT_DIR)
    sh("git", "checkout", BRANCH, cwd=PROJECT_DIR)
    sh("git", "pull", "--ff-only", cwd=PROJECT_DIR)

os.chdir(PROJECT_DIR)
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".")

## Secrets and remote storage

Set these in Colab secrets or environment before running: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`, `ARTIFACT_DVC_REMOTE`, and optionally `ARTIFACT_REMOTE_URI`. API keys are not written to notebook outputs.

In [ ]:
try:
    from google.colab import userdata
except Exception:
    userdata = None


def find_local_env() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / ".env"
        if candidate.exists():
            return candidate
    return None


def load_local_env(path: str | Path | None = None):
    env_path = Path(path) if path is not None else find_local_env()
    print("Notebook cwd:", Path.cwd())
    if env_path is None or not env_path.exists():
        print("Local .env not found")
        return
    print("Loading local env from:", env_path)
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value and not os.getenv(key):
            os.environ[key] = value


# In local VS Code/Jupyter, Colab Secrets are unavailable, so read repo .env.
if userdata is None:
    load_local_env()


def set_secret(name, default=None):
    if os.getenv(name):
        return
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return
    if default is not None:
        os.environ[name] = default


set_secret("AWS_ACCESS_KEY_ID")
set_secret("AWS_SECRET_ACCESS_KEY")
set_secret("AWS_REGION", "eu-north-1")
set_secret("ARTIFACT_DVC_REMOTE", "artifact_s3")
set_secret("ARTIFACT_REMOTE_URI")

print("DVC remote:", os.getenv("ARTIFACT_DVC_REMOTE"))
print("AWS region:", os.getenv("AWS_REGION"))
print("Artifact remote URI set:", bool(os.getenv("ARTIFACT_REMOTE_URI")))

In [ ]:
required_env = [
    "ARTIFACT_REMOTE_URI",
    "ARTIFACT_DVC_REMOTE",
    "AWS_REGION",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
]
print("artifact_pull env:")
for name in required_env:
    value = os.getenv(name)
    if name.startswith("AWS_") and value:
        display_value = "set"
    else:
        display_value = value or "MISSING"
    print(f"  {name}: {display_value}")

missing = [
    name
    for name in ("ARTIFACT_REMOTE_URI", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
    if not os.getenv(name)
]
if missing:
    raise RuntimeError(
        "Missing Colab secrets/env vars for artifact_pull: "
        + ", ".join(missing)
        + ". Add them in Colab Secrets and enable notebook access, then rerun the setup cell."
    )

cmd = [sys.executable, "-m", "medical_rag_reranker.commands", "artifact_pull"]
print("+", " ".join(cmd))
proc = subprocess.run(cmd, text=True, capture_output=True)
if proc.stdout:
    print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f"artifact_pull failed with exit code {proc.returncode}")

## Train Cross-Encoder reranker

This uses the full dataloaders by setting `limit_train_batches=1.0` and `limit_val_batches=1.0`. Tune batch size and epochs depending on the GPU.

In [ ]:
import json
from pathlib import Path

required_processed = [
    Path("data/processed/qa.jsonl"),
    Path("data/processed/corpus.jsonl"),
    Path("data/processed/splits.json"),
]
missing_processed = [path for path in required_processed if not path.exists()]
if missing_processed:
    raise RuntimeError(
        "Missing processed artifacts for reranker training: "
        + ", ".join(str(path) for path in missing_processed)
        + ". Run artifact_pull before training."
    )

print("Processed artifacts found:")
for path in required_processed:
    print(f"  {path}: {path.stat().st_size} bytes")

overrides = [
    "data.use_dvc=false",
    "data.prefer_prepared_artifacts=true",
    "data.processed_dir=data/processed",
    f"model.max_length={int(os.getenv('RERANKER_MAX_LENGTH', '192'))}",
    f"model.negatives_per_query={int(os.getenv('RERANKER_NEGATIVES_PER_QUERY', '2'))}",
    f"train.max_epochs={int(os.getenv('RERANKER_EPOCHS', '2'))}",
    f"train.batch_size={int(os.getenv('RERANKER_BATCH_SIZE', '16'))}",
    f"train.num_workers={int(os.getenv('RERANKER_NUM_WORKERS', '4'))}",
    f"train.hard_negative_pool_size={int(os.getenv('RERANKER_HARD_NEGATIVE_POOL_SIZE', '10'))}",
    f"train.limit_train_batches={os.getenv('RERANKER_LIMIT_TRAIN_BATCHES', '1.0')}",
    f"train.limit_val_batches={os.getenv('RERANKER_LIMIT_VAL_BATCHES', '1.0')}",
    "train.accelerator=gpu",
    "train.devices=1",
    f"train.enable_progress_bar={os.getenv('RERANKER_ENABLE_PROGRESS_BAR', 'true').lower()}",
    f"train.progress_refresh_rate={int(os.getenv('RERANKER_PROGRESS_REFRESH_RATE', '100'))}",
    f"train.log_every_n_steps={int(os.getenv('RERANKER_LOG_EVERY_N_STEPS', '100'))}",
    f"logging.tracking_uri={MLFLOW_TRACKING_URI}",
    f"logging.run_name=reranker_{RUN_ID}",
]

print("Starting Cross-Encoder reranker training", flush=True)
print("Overrides:", json.dumps(overrides, indent=2), flush=True)

run_streaming(
    sys.executable,
    "-u",
    "-m",
    "medical_rag_reranker.commands",
    "train",
    "--overrides",
    json.dumps(overrides),
)

## Persist artifacts

Copy or move the resulting checkpoint into the experiment artifact directory if the training command writes it elsewhere, then push the experiment folder and DVC metadata to S3.

In [ ]:
import shutil

RERANKER_DIR.mkdir(parents=True, exist_ok=True)
target = RERANKER_DIR / "best.ckpt"
search_roots = [PROJECT_DIR]
drive_mlruns = Path("/content/drive/MyDrive/medical-rag-reranker-colab/mlruns")
if drive_mlruns.exists():
    search_roots.append(drive_mlruns)

ckpts = []
for root in search_roots:
    ckpts.extend(root.rglob("*.ckpt"))
ckpts = sorted(
    [p for p in ckpts if p.resolve() != target.resolve()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

print("Found checkpoints:")
for p in ckpts[:10]:
    print(p)

if not ckpts:
    raise RuntimeError("No reranker .ckpt files found after training.")

best_ckpt = ckpts[0]
shutil.copy2(best_ckpt, target)
os.environ["GENERATION_RERANKER_CHECKPOINT_PATH"] = str(target)

print("Copied checkpoint from:", best_ckpt)
print("Stable reranker checkpoint:", target)
print("Expected reranker artifact dir:", RERANKER_DIR.resolve())

# Push all current runtime artifacts plus experiment outputs.
sh(
    sys.executable,
    "-m",
    "medical_rag_reranker.commands",
    "artifact_push",
    "--include",
    f"artifacts/experiments/{RUN_ID}/**/*",
)